In [31]:
%load_ext autoreload
%autoreload 2

import sklearn
import scipy 
import numpy as np
import pandas as pd
import os
import sys
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
sys.path.append(os.path.abspath("src"))
import jdcoot
import math
from scipy.io import loadmat

from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
from jdcoot.models.discrete_partial_jdcoot2 import discrete_partial_jdcoot2 # fonction pour verifier que jdcoot donne les memes perfs que coot quand la boucle n'est que sur le transport
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
from jdcoot.models.discrete_semisupervised_coot import discrete_semisupervised_coot
from jdcoot.models.discrete_partial_coot import discrete_partial_coot
from jdcoot.models.discrete_partial_coot2 import discrete_partial_coot2 # 2 coot semi supervisé
from jdcoot.models.discrete_partial_jdcoot3 import discrete_partial_jdcoot3 # 2 coot plus JDCOOT
from jdcoot.models.discrete_semisupervised_reference import discrete_semisupervised_reference
from jdcoot.models.discrete_partial_reference import discrete_partial_reference
from jdcoot.utils import xcolumns, continuous_classifiers, continuous_accuracy
from sklearn.model_selection import train_test_split

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# les donnees caffeNet GoogleNet

In [3]:
featuresToUse = ["CaffeNet4096", "GoogleNet1024"] 
#featuresToUse = ["CaffeNet4096", "CaffeNet4096"] 
sourceDomainName = ['caltech10'] #['caltech10','amazon','webcam']
targetDomainName = ['caltech10'] #['caltech10','amazon','webcam']

min_max_scaler = sklearn.preprocessing.MinMaxScaler()
# Collab
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[0],
                                                 "caltech10" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
S_data = [feat, labels]
S_nClass = len(np.unique(labels)) # nb de class in source data
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[1],
                                                 "amazon" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
T_data = [feat, labels]
T_nClass = len(np.unique(labels)) # nb de class in target data

source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1


In [3]:
results = []
numRepetitions = 10
alpha =1.5# hyperparamètre devant la loss a été optimé
prop_target = 0.005
algo = "sinkhorn"
reg = 1


In [4]:
a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
S = source.iloc[a, :].reset_index(drop=True)
T = target.iloc[b, :].reset_index(drop=True)

In [5]:
S.shape, T.shape, S_test.shape, T_test.shape

((899, 4097), (767, 1025), (224, 4097), (191, 1025))

# Non supervisé

In [7]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(
                S, T, S_test, T_test,algo="sinkhorn",reg=1,batch_size=20,alpha =1.5
            )
test_target

Delta: 0.02092798685283678 	  Loss: 2.1473654086839193 	 Accuracy: 0.20860495436766624
Delta: 0.019073847854404635 	  Loss: 2.048302812754046 	 Accuracy: 0.37157757496740546
Delta: 0.01676452405183621 	  Loss: 1.8677940006397904 	 Accuracy: 0.4876140808344198
Delta: 0.014049839000854727 	  Loss: 1.7163144169832814 	 Accuracy: 0.41851368970013036
Delta: 0.01203145438126765 	  Loss: 1.6360994358998124 	 Accuracy: 0.39374185136897
Delta: 0.010175359826428095 	  Loss: 1.5970084424659972 	 Accuracy: 0.3833116036505867
Delta: 0.008718547889898237 	  Loss: 1.5790902327283272 	 Accuracy: 0.3741851368970013
Delta: 0.008127365247757617 	  Loss: 1.5700033860280627 	 Accuracy: 0.36766623207301175
Delta: 0.007497062249503367 	  Loss: 1.5649376118408074 	 Accuracy: 0.3376792698826597


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.007013439020467204 	  Loss: 1.5619345604898287 	 Accuracy: 0.3363754889178618
Delta: 0.006765096592841762 	  Loss: 1.559713940518212 	 Accuracy: 0.3376792698826597
Delta: 0.00524886880493173 	  Loss: 1.5585495273837306 	 Accuracy: 0.3376792698826597
Delta: 0.004513107968689477 	  Loss: 1.5577329387462076 	 Accuracy: 0.3376792698826597
Delta: 0.004889221528666393 	  Loss: 1.557083600310453 	 Accuracy: 0.3376792698826597
Delta: 0.0044407014412937455 	  Loss: 1.5567998672979082 	 Accuracy: 0.3376792698826597
Delta: 0.004125334229077259 	  Loss: 1.55655295715381 	 Accuracy: 0.34028683181225555
Delta: 0.004656910069616502 	  Loss: 1.5563580600682725 	 Accuracy: 0.3389830508474576
Delta: 0.0022984233248476275 	  Loss: 1.5562576866919726 	 Accuracy: 0.3389830508474576
Delta: 0.003965730686381726 	  Loss: 1.5560767563588846 	 Accuracy: 0.34028683181225555
Delta: 0.0012283781359316453 	  Loss: 1.5560912717845174 	 Accuracy: 0.34028683181225555
Delta: 0.001000441287283334 	  Loss: 1.556

np.float64(0.3089005235602094)

In [7]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_coot(
                S, T, S_test, T_test,algo="sinkhorn",reg=1,batch_size=20
            )
test_target

Delta:       0.0156335 	 Loss:       2.2046902
Delta:       0.0209275 	 Loss:       2.1683288
Delta:       0.0187107 	 Loss:       2.1067086
Delta:       0.0139696 	 Loss:       2.0724738
Delta:       0.0105210 	 Loss:       2.0659279
Delta:       0.0099668 	 Loss:       2.0633529
Delta:       0.0097964 	 Loss:       2.0600302
Delta:       0.0083934 	 Loss:       2.0574664
Delta:       0.0069205 	 Loss:       2.0561957
Delta:       0.0060637 	 Loss:       2.0554801
Delta:       0.0038115 	 Loss:       2.0552763
Delta:       0.0029421 	 Loss:       2.0551820
Delta:       0.0025912 	 Loss:       2.0551839
Delta:       0.0021900 	 Loss:       2.0551567
Delta:       0.0026604 	 Loss:       2.0551325
Delta:       0.0010400 	 Loss:       2.0551285
Delta:       0.0018294 	 Loss:       2.0551359


KeyboardInterrupt: 

# semi supervisé

In [4]:
n_target = len(T.Z)
l_train, l_test = train_test_split(
        np.arange(n_target),
        train_size=0.01,
    )

In [5]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_jdcoot(
                S, T, S_test, T_test,l_train, l_test,algo="sinkhorn",reg=1,batch_size=20,alpha =0.001         
            )
test_target

Delta: 0.015934172251934338 	 Loss: 2.197176181582239 	 Accuracy: 0.10394736842105264
Delta: 0.020814120647066645 	 Loss: 2.134796064429104 	 Accuracy: 0.3171052631578947
Delta: 0.01565944805950562 	 Loss: 2.095024680291388 	 Accuracy: 0.3605263157894737
Delta: 0.012760541781771474 	 Loss: 2.0791209707700133 	 Accuracy: 0.39473684210526316


KeyboardInterrupt: 

In [7]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_coot(
                S, T, S_test, T_test,l_train, l_test,algo="sinkhorn",reg=1,batch_size=20            
            )
test_target

Delta:       0.0159570 	 Loss:       2.1930081
Delta:       0.0203894 	 Loss:       2.0893501
Delta:       0.0137277 	 Loss:       2.0483110
Delta:       0.0100523 	 Loss:       2.0395606
Delta:       0.0067977 	 Loss:       2.0375050
Delta:       0.0027580 	 Loss:       2.0371000
Delta:       0.0006959 	 Loss:       2.0370704
Delta:       0.0000003 	 Loss:       2.0370701
Delta:       0.0000000 	 Loss:       2.0370701
converged at iter  8


np.float64(0.6701570680628273)

In [8]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_reference(
                S, T, S_test, T_test,l_train, l_test,algo="sinkhorn",reg=1,batch_size=20                      
            )
test_target

np.float64(0.15657894736842104)

# partial

In [ ]:
# label observé aléatoire dans source et target

array([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])

In [5]:
prop_source=0.5
prop_target=0.005 
n_target = len(T.Z)
n_source = len(S.Z)
l_source_train, l_source_test = train_test_split(
        np.arange(n_source),
        train_size=prop_source
    )

l_target_train, l_target_test = train_test_split(
        np.arange(n_target),
        train_size=prop_target
    )

In [54]:
l_target_train

array([ 57,  46, 399])

In [ ]:
# lun label dans chaque classe dans source et target

In [14]:

# SOURCE
l_source_train = []
for z in np.unique(S['Z']):
    idx = np.where(S['Z'] == z)[0]
    chosen = np.random.choice(idx, size=1, replace=False)
    l_source_train.extend(chosen)

l_source_train = np.array(l_source_train)

# le reste en test
l_source_test = np.setdiff1d(np.arange(len(S['Z'])), l_source_train)


# TARGET
l_target_train = []
for z in np.unique(T['Z']):
    idx = np.where(T['Z'] == z)[0]
    chosen = np.random.choice(idx, size=1, replace=False)
    l_target_train.extend(chosen)

l_target_train = np.array(l_target_train)

l_target_test = np.setdiff1d(np.arange(len(T['Z'])), l_target_train)

In [ ]:
# les labels 0,1,2,3,4 observés dans source et les labels 5,6,7,8,9 observés dans source

In [23]:
import numpy as np

# ===== SOURCE =====
l_source_train = np.where(np.isin(S['Z'], [0,1,2,3,4]))[0]
l_source_test  = np.where(np.isin(S['Z'], [5,6,7,8,9]))[0]

# ===== TARGET =====
l_target_train = np.where(np.isin(T['Z'], [5,6,7,8,9]))[0]
l_target_test  = np.where(np.isin(T['Z'], [0,1,2,3,4]))[0]

In [24]:
pure_source, pure_target, test_source, test_target = discrete_partial_coot(
                S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo="sinkhorn",reg=1,batch_size=20            
            )
test_target

 Accuracy: 0.0


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/backend.py:1197: RuntimeWarning: invalid value encountered in dot
  return np.dot(a, b)
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


Delta:       0.0167682 	 Loss:       2.9427768
Delta:       0.0210473 	 Loss:       2.9207570
Delta:       0.0177192 	 Loss:       2.9147850
Delta:       0.0139118 	 Loss:       2.9121008
Delta:       0.0098749 	 Loss:       2.9114038
Delta:       0.0067772 	 Loss:       2.9112590
Delta:       0.0032583 	 Loss:       2.9112386
Delta:       0.0013814 	 Loss:       2.9112392
Delta:       0.0000001 	 Loss:       2.9112390
Delta:       0.0000000 	 Loss:       2.9112390
converged at iter  9
Ts sum: 0.07009472472107571
Ts max: 8.420231176832568e-07
Ts min: 0.0


np.float64(0.387434554973822)

In [25]:
pure_source, pure_target, test_source, test_target = discrete_partial_coot2(
                S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo="sinkhorn",reg=1,batch_size=20          

            )
test_target

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/backend.py:1197: RuntimeWarning: invalid value encountered in dot
  return np.dot(a, b)
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


Delta:       0.0172118 	 Loss:       2.9961616
Delta:       0.0210848 	 Loss:       2.9728821
Delta:       0.0179364 	 Loss:       2.9653513
Delta:       0.0126613 	 Loss:       2.9604837
Delta:       0.0100553 	 Loss:       2.9599071
Delta:       0.0087121 	 Loss:       2.9597040
Delta:       0.0082257 	 Loss:       2.9595471
Delta:       0.0063323 	 Loss:       2.9594523
Delta:       0.0051115 	 Loss:       2.9594136
Delta:       0.0042163 	 Loss:       2.9593993
Delta:       0.0029511 	 Loss:       2.9593898
Delta:       0.0029910 	 Loss:       2.9593865
Delta:       0.0025382 	 Loss:       2.9593803
Delta:       0.0034190 	 Loss:       2.9593751
Delta:       0.0044365 	 Loss:       2.9593640
Delta:       0.0043001 	 Loss:       2.9593517
Delta:       0.0057696 	 Loss:       2.9593325
Delta:       0.0062844 	 Loss:       2.9593029
Delta:       0.0063694 	 Loss:       2.9592585
Delta:       0.0057281 	 Loss:       2.9592286
Delta:       0.0051580 	 Loss:       2.9592113
Delta:       

np.float64(0.5549738219895288)

In [26]:
pure_source, pure_target, test_source, test_target = discrete_partial_jdcoot2(
                S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo="sinkhorn",reg=1,batch_size=20,alpha =2
            )
test_target

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/backend.py:1197: RuntimeWarning: invalid value encountered in dot
  return np.dot(a, b)
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


converged at iter  9
Delta: 0.0 	  Loss: 2.911239040514798 	 Accuracy: 0.1989389920424403
Gs sum: 0.07009472472107571
Gs max: 8.420231176832568e-07
Gs min: 0.0


np.float64(0.5445026178010471)

In [ ]:
pure_source, pure_target, test_source, test_target = discrete_partial_jdcoot(
                S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo="sinkhorn",reg=1,batch_size=20,alpha =1.5
            )
test_target

 Accuracy: 0.0


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/backend.py:1197: RuntimeWarning: invalid value encountered in dot
  return np.dot(a, b)
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


Delta: 0.01676821582103159 	  Loss: 2.942776814040057 	 Accuracy: 0.14854111405835543
Delta: 0.024029270971502942 	  Loss: 2.0379202721744027 	 Accuracy: 0.23607427055702918
Delta: 0.022627640649863494 	  Loss: 1.889739487678472 	 Accuracy: 0.18037135278514588
Delta: 0.01930552521184871 	  Loss: 1.8526860160131398 	 Accuracy: 0.1883289124668435
Delta: 0.019376936838363244 	  Loss: 1.8109803996128973 	 Accuracy: 0.21750663129973474


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.013322614923139246 	  Loss: 1.8092702538227547 	 Accuracy: 0.22015915119363394
Delta: 0.017374115618540034 	  Loss: 1.8045224810283282 	 Accuracy: 0.22015915119363394
Delta: 0.016932893453113353 	  Loss: 1.7911382786210273 	 Accuracy: 0.22015915119363394
Delta: 0.009962074988397001 	  Loss: 1.7887983578203461 	 Accuracy: 0.22811671087533156
Delta: 0.013819763612450426 	  Loss: 1.7863970144258545 	 Accuracy: 0.22015915119363394
Delta: 0.014303067198630351 	  Loss: 1.7904710523323568 	 Accuracy: 0.23342175066312998
Delta: 0.01222964374813747 	  Loss: 1.7906924414320333 	 Accuracy: 0.2413793103448276
Delta: 0.013899052841575174 	  Loss: 1.7935809393336963 	 Accuracy: 0.2440318302387268
Delta: 0.015958054454008695 	  Loss: 1.772281575818183 	 Accuracy: 0.2440318302387268
Delta: 0.011281497627432224 	  Loss: 1.7732647817207385 	 Accuracy: 0.2519893899204244
Delta: 0.010388061036680583 	  Loss: 1.7710919178574627 	 Accuracy: 0.2440318302387268
Delta: 0.009642028162976308 	  Loss: 1.

np.float64(0.5392670157068062)

In [32]:
pure_source, pure_target, test_source, test_target = discrete_partial_jdcoot3(
                S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo="sinkhorn",reg=1,batch_size=20,alpha =1.5
            )
test_target

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/backend.py:1197: RuntimeWarning: invalid value encountered in dot
  return np.dot(a, b)
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


Delta:       0.0172118 	 Loss:       2.9961616
Delta:       0.0210848 	 Loss:       2.9728821
Delta:       0.0179364 	 Loss:       2.9653513
Delta:       0.0126613 	 Loss:       2.9604837
Delta:       0.0100553 	 Loss:       2.9599071
Delta:       0.0087121 	 Loss:       2.9597040
Delta:       0.0082257 	 Loss:       2.9595471
Delta:       0.0063323 	 Loss:       2.9594523
Delta:       0.0051115 	 Loss:       2.9594136
Delta:       0.0042163 	 Loss:       2.9593993
Delta:       0.0029511 	 Loss:       2.9593898
Delta:       0.0029910 	 Loss:       2.9593865
Delta:       0.0025382 	 Loss:       2.9593803
Delta:       0.0034190 	 Loss:       2.9593751
Delta:       0.0044365 	 Loss:       2.9593640
Delta:       0.0043001 	 Loss:       2.9593517
Delta:       0.0057696 	 Loss:       2.9593325
Delta:       0.0062844 	 Loss:       2.9593029
Delta:       0.0063694 	 Loss:       2.9592585
Delta:       0.0057281 	 Loss:       2.9592286
Delta:       0.0051580 	 Loss:       2.9592113
Delta:       

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.017545132109656324 	  Loss: 1.759618660778885 	 Accuracy: 0.5305039787798409
Delta: 0.015947244869049776 	  Loss: 1.7731743090533678 	 Accuracy: 0.493368700265252
Delta: 0.012048392355310442 	  Loss: 1.7693218029860152 	 Accuracy: 0.5119363395225465
Delta: 0.013945233890484177 	  Loss: 1.7675080233148643 	 Accuracy: 0.5092838196286472
Delta: 0.012913387432786168 	  Loss: 1.7499110868803163 	 Accuracy: 0.5013262599469496
Delta: 0.013622798611301369 	  Loss: 1.7526929084261371 	 Accuracy: 0.5013262599469496
Delta: 0.013261266318814855 	  Loss: 1.7511450383456524 	 Accuracy: 0.5039787798408488
Delta: 0.009323109054997163 	  Loss: 1.750101601637323 	 Accuracy: 0.506631299734748
Delta: 0.012664368203695394 	  Loss: 1.7523538403662524 	 Accuracy: 0.493368700265252
Delta: 0.008445867512633279 	  Loss: 1.7508889025665053 	 Accuracy: 0.5039787798408488
Delta: 0.01436061917418531 	  Loss: 1.750463563422895 	 Accuracy: 0.4986737400530504
Delta: 0.0118457714378724 	  Loss: 1.7399081075293

KeyboardInterrupt: 

In [30]:
pure_source, pure_target, test_source, test_target = discrete_partial_reference(
                S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test
            )
test_target


np.float64(0.5130890052356021)

In [ ]:
import numpy as np
# Exemple de valeurs à tester pour alpha
alpha_values = np.linspace(1, 3, 6)  # 0, 0.1, 0.2, ..., 1.0
best_alpha = None
best_score = -np.inf  # ou 0 selon ta métrique
results = []
numRepetitions = 1
for repe in range(numRepetitions):
   
        a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
        b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

        S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
        T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
        S = source.iloc[a, :].reset_index(drop=True)
        T = target.iloc[b, :].reset_index(drop=True)
        n_target = len(T.Z)
        n_source = len(S.Z)
        l_source_train, l_source_test = train_test_split(
            np.arange(n_source),
            train_size=prop_source
        )

        l_target_train, l_target_test = train_test_split(
            np.arange(n_target),
            train_size=prop_target
            )
        for a in alpha_values:
            pure_source, pure_target, test_source, test_target = \
            discrete_partial_jdcoot(S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo="sinkhorn",reg=1,batch_size=20,alpha =a)

            score = test_target  
    
            if score > best_score:
                best_score = score
                best_alpha = a

        results.append({
         "repetition": repe,
         "recoding": "jdcoot",
         "learning": "unsupervised",
         "alpha": best_alpha,
     })

   

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "alpha"],
        as_index=False
    )
    .agg(
        alpha_mean=("alpha", "mean"),
    )
)

df_summary

df_summary.to_excel("results_mean.xlsx", index=False)


Delta: 0.017223660338058226 	  Loss: 2.0444923080506943 	 Accuracy: 0.18586387434554974
Delta: 0.018464328934951865 	  Loss: 2.0445552692213846 	 Accuracy: 0.2212041884816754


In [6]:
df_summary.to_excel("results_mean.xlsx", index=False)

In [8]:
df_results 

,repetition,recoding,learning,alpha
0,0,jdcoot,unsupervised,1.9
1,1,jdcoot,unsupervised,1.9
2,2,jdcoot,unsupervised,1.9
3,3,jdcoot,unsupervised,1.9
4,4,jdcoot,unsupervised,1.9
5,5,jdcoot,unsupervised,1.9
6,6,jdcoot,unsupervised,1.9
7,7,jdcoot,unsupervised,1.0
8,8,jdcoot,unsupervised,1.0
9,9,jdcoot,unsupervised,1.0
